## 1. Import app and extensions

In [ ]:
from typing import List
from Project.core.app import app
from Project.core.extensions import db
from Project.config import config, Config
from Project.services import pf, geodb
from Project.services.petfinder.petfinder_types import AnimalReqParams
from Project.utils import dynamic_rate_limit  # import custom dynamic rate limit
from Project.models import AnimalCity
from Project.schemas.common import AddressSchema, CitySchema
from Project.schemas.animals import (
    AnimalResponseSchema,
    AnimalCityJoinSchema,
    AnimalListResponseSchema,
)


# main HTTP function for PetFinder
@dynamic_rate_limit
def get_pf_data(endpoint, params: AnimalReqParams) -> AnimalListResponseSchema:
    data = pf._get_request(endpoint=endpoint, params=params)
    data.raise_for_status()

    return data.json()


# get geolocation from GeoDb
@dynamic_rate_limit
def scrape_geolocation(country: str, state: str):
    return geodb.get_state_geolocation(state=state, country=country)


# get city information from GeoDb
@dynamic_rate_limit
def scrape_city_details(city_name) -> dict:
    return geodb.get_city_details(city_identifier=city_name)


with app.app_context().push():
    db.create_all()

In [ ]:
import logging
from logging.config import dictConfig
from Project.config import Config
from typing import List, Callable, Any, Optional
import time

# Configure logging
logging.config.dictConfig(Config.get_logging_config())
logger = logging.getLogger(__name__)


@dynamic_rate_limit
def scrape_animals_and_cities(
    locations: List[dict],
    db: Any,
    check_cities_db_func: Callable[[str, Any], bool],
    check_animalcities_db_func: Callable[[str, Any], bool],
    get_pf_data_func: Callable[[str, Any], Any],
    scrape_geolocation_func: Callable[[str, str], Any],
    save_to_db_func: Callable[[Any, Any, Any], None],
) -> None:
    """
    Process a list of locations, making HTTP requests for both animals and cities,
    and saving data to the database using the AnimalCity joint table.

    :param locations: List of dictionaries containing location information (country, state, city)
    :param db: Database object (Flask SQLAlchemy object)
    :param check_cities_db_func: Function to check if a cities table entry exists in the database
    :param check_animalcities_db_func: Function to check if an animal & cities joint entry exists in the database
    :param get_pf_data_func: Function to make HTTP requests to PetFinder API
    :param scrape_geolocation_func: Function to get geolocation data for cities
    :param save_to_db_func: Function to save data to the database using AnimalCity
    """
    for location in locations:
        try:
            country = location.get("country")
            state = location.get("state")
            city = location.get("city")

            # Check if the city is already in the database
            if check_cities_db_func(city, db):
                logger.info(f"City {city} already exists in the database. Skipping.")
                continue

            # Get geolocation data for the city
            city_data = scrape_geolocation_func(country, state)

            # Get animal data from PetFinder
            animal_params = AnimalReqParams(location=f"{city}, {state}")
            animal_data = get_pf_data_func("animals", animal_params)

            # Check if the city is already in the database
            if not check_animalcities_db_func(
                city, state, country, animal_data.get("animals.id", None),
            ):

                # Combine city and animal data
                combined_data = {
                    "city": city_data,
                    "animals": animal_data.get("animals", []),
                }

                # Save data to database using AnimalCity
                save_to_db_func(combined_data, db)

                logger.info(
                    f"Successfully processed and saved data for {city}, {state}"
                )

            # Add a small delay to avoid overwhelming the server
            time.sleep(3)

        except Exception as e:
            logger.error(f"Error processing {city}, {state}: {str(e)}")

    logger.info("Finished processing all locations")

These dictionaries contain lists of major cities for each state/province in Canada and the United States, likely for use in a data scraping or processing task related to animal content in North American cities.

In [ ]:
# Canadian Provinces and Territories
canada = {
    "Ontario": ["Toronto", "Ottawa", "London", "Waterloo", "Niagara Falls"],
    "Quebec": ["Montreal", "Quebec City", "Gatineau"],
    "British Columbia": ["Vancouver", "Victoria", "Whistler"],
    "Alberta": ["Calgary", "Edmonton", "Banff"],
    "Manitoba": ["Winnipeg", "Brandon", "Churchill"],
    "Saskatchewan": ["Saskatoon", "Regina", "Moose Jaw"],
    "Nova Scotia": ["Halifax", "Sydney", "Dartmouth"],
    "New Brunswick": ["Fredericton", "Saint John", "Moncton"],
    "Newfoundland and Labrador": ["St. John's", "Corner Brook", "Labrador City"],
    "Prince Edward Island": ["Charlottetown", "Summerside", "Stratford"],
    "Northwest Territories": ["Yellowknife", "Inuvik", "Hay River"],
    "Yukon": ["Whitehorse", "Dawson City", "Watson Lake"],
    "Nunavut": ["Iqaluit", "Rankin Inlet", "Arviat"],
}

# United States
usa = {
    "United States of America": {
        "Alabama": ["Birmingham", "Montgomery", "Mobile"],
        "Alaska": ["Anchorage", "Fairbanks", "Juneau"],
        "Arizona": ["Phoenix", "Tucson", "Sedona"],
        "Arkansas": ["Little Rock", "Fayetteville", "Hot Springs"],
        "California": ["Los Angeles", "San Francisco", "San Diego"],
        "Colorado": ["Denver", "Colorado Springs", "Boulder"],
        "Connecticut": ["Hartford", "New Haven", "Stamford"],
        "Delaware": ["Wilmington", "Dover", "Newark"],
        "Florida": ["Miami", "Orlando", "Tampa"],
        "Georgia": ["Atlanta", "Savannah", "Augusta"],
        "Hawaii": ["Honolulu", "Hilo", "Lahaina"],
        "Idaho": ["Boise", "Idaho Falls", "Coeur d'Alene"],
        "Illinois": ["Chicago", "Springfield", "Naperville"],
        "Indiana": ["Indianapolis", "Fort Wayne", "Bloomington"],
        "Iowa": ["Des Moines", "Iowa City", "Cedar Rapids"],
        "Kansas": ["Wichita", "Kansas City", "Topeka"],
        "Kentucky": ["Louisville", "Lexington", "Frankfort"],
        "Louisiana": ["New Orleans", "Baton Rouge", "Lafayette"],
        "Maine": ["Portland", "Augusta", "Bar Harbor"],
        "Maryland": ["Baltimore", "Annapolis", "Ocean City"],
        "Massachusetts": ["Boston", "Cambridge", "Salem"],
        "Michigan": ["Detroit", "Ann Arbor", "Grand Rapids"],
        "Minnesota": ["Minneapolis", "St. Paul", "Duluth"],
        "Mississippi": ["Jackson", "Biloxi", "Oxford"],
        "Missouri": ["Kansas City", "St. Louis", "Springfield"],
        "Montana": ["Billings", "Missoula", "Bozeman"],
        "Nebraska": ["Omaha", "Lincoln", "Grand Island"],
        "Nevada": ["Las Vegas", "Reno", "Carson City"],
        "New Hampshire": ["Manchester", "Concord", "Portsmouth"],
        "New Jersey": ["Newark", "Jersey City", "Atlantic City"],
        "New Mexico": ["Albuquerque", "Santa Fe", "Taos"],
        "New York": ["New York City", "Buffalo", "Albany"],
        "North Carolina": ["Charlotte", "Raleigh", "Asheville"],
        "North Dakota": ["Fargo", "Bismarck", "Grand Forks"],
        "Ohio": ["Columbus", "Cleveland", "Cincinnati"],
        "Oklahoma": ["Oklahoma City", "Tulsa", "Norman"],
        "Oregon": ["Portland", "Eugene", "Bend"],
        "Pennsylvania": ["Philadelphia", "Pittsburgh", "Gettysburg"],
        "Rhode Island": ["Providence", "Newport", "Warwick"],
        "South Carolina": ["Charleston", "Columbia", "Myrtle Beach"],
        "South Dakota": ["Sioux Falls", "Rapid City", "Pierre"],
        "Tennessee": ["Nashville", "Memphis", "Chattanooga"],
        "Texas": ["Houston", "Austin", "Dallas"],
        "Utah": ["Salt Lake City", "Park City", "Moab"],
        "Vermont": ["Burlington", "Montpelier", "Stowe"],
        "Virginia": ["Richmond", "Virginia Beach", "Charlottesville"],
        "Washington": ["Seattle", "Spokane", "Olympia"],
        "West Virginia": ["Charleston", "Morgantown", "Huntington"],
        "Wisconsin": ["Milwaukee", "Madison", "Green Bay"],
        "Wyoming": ["Cheyenne", "Jackson", "Casper"],
    }
}

In [ ]:
# Flatten dict of cities before API calls
alternating_canada_cities = geodb.alternate_cities(canada)
alternating_usa_cities = geodb.alternate_cities(usa)

canadian_animal_city_data = scrape_animals_and_cities(
    locations=alternating_canada_cities,
    db=db,
    check_cities_db_func=AnimalCity.check_city_animals_exist,
    check_animalcities_db_func=AnimalCity.check_city_animals_exist,
    save_to_db_func=AnimalCity.create_from_combined_dict,
    scrape_geolocation_func=scrape_geolocation,
)